# Version 4 - Model Training

This notebook mirrors the training workflow used by the version 4 scripts. It loads the rainfall dataset, adds a small set of engineered features, compares a few regression models, and saves the best one.

In [ ]:
from pathlib import Path
import json
import math

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "Indian Rainfall Dataset District-wise Daily Measurements.csv"
MODEL_PATH = PROJECT_ROOT / "rainfall_model.pkl"
METRICS_PATH = PROJECT_ROOT / "rainfall_model_metrics.json"

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DAY_COLUMNS = [
    "1st", "2nd", "3rd", "4th", "5th", "6th", "7th", "8th", "9th",
    "10th", "11th", "12th", "13th", "14th", "15th", "16th", "17th",
    "18th", "19th", "20th", "21st", "22nd", "23rd", "24th", "25th",
    "26th", "27th", "28th", "29th", "30th",
]


def season(month: int) -> str:
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Summer"
    if month in [6, 7, 8, 9]:
        return "Monsoon"
    return "PostMonsoon"


def load_data(path: Path = DATA_PATH) -> pd.DataFrame:
    return pd.read_csv(path, sep=";")


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.drop_duplicates().copy()
    return cleaned.fillna(0)


def add_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    engineered = df.copy()
    engineered["month"] = pd.to_numeric(engineered["month"], errors="coerce").fillna(0).astype(int)
    engineered["Season"] = engineered["month"].apply(season)
    engineered["TotalRainfall"] = engineered[DAY_COLUMNS].sum(axis=1)
    engineered["AverageRainfall"] = engineered[DAY_COLUMNS].mean(axis=1)
    return engineered


df = add_feature_engineering(clean_data(load_data()))
df.head()

In [ ]:
def split_features_target(df: pd.DataFrame, target_column: str = "31st"):
    features = df.drop(columns=[target_column])
    target = df[target_column]
    return features, target


def build_preprocessor(feature_frame: pd.DataFrame) -> ColumnTransformer:
    categorical_features = ["state", "district", "Season"]
    numeric_features = [column for column in feature_frame.columns if column not in categorical_features]

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                ]),
                numeric_features,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical_features,
            ),
        ]
    )


features, target = split_features_target(df, target_column="31st")
x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
)

preprocessor = build_preprocessor(x_train)

candidate_models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1,
    ),
    "gradient_boosting": GradientBoostingRegressor(random_state=42),
}

results = {}
best_model = None
best_name = None
best_mae = float("inf")

for name, estimator in candidate_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    pipeline.fit(x_train, y_train)
    predictions = pipeline.predict(x_test)
    mse = mean_squared_error(y_test, predictions)
    metrics = {
        "mae": mean_absolute_error(y_test, predictions),
        "mse": mse,
        "rmse": math.sqrt(mse),
        "r2": r2_score(y_test, predictions),
    }
    results[name] = metrics
    if metrics["mae"] < best_mae:
        best_mae = metrics["mae"]
        best_name = name
        best_model = pipeline

best_name, results

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)

with METRICS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "best_model": best_name,
            "results": results,
        },
        handle,
        indent=2,
    )

sample_predictions = best_model.predict(x_test.head(5))
{
    "best_model": best_name,
    "sample_predictions": sample_predictions.tolist(),
    "actual_values": y_test.head(5).tolist(),
}